# Notebook 36 — Continuous Learning

Agents that improve from feedback over time.

| Component | Role |
|---|---|
| `FeedbackEntry` | (input, output, score, correction) record |
| `FeedbackStore` | in-memory or SQLite feedback log |
| `ExperienceReplay` | replay buffer with prioritised sampling |
| `FewShotSelector` | pick best examples by similarity + score |
| `AdaptivePrompt` | prompt that enriches itself from feedback |
| `OnlineLearner` | epsilon-greedy bandit over prompt variants |
| `RewardSignal` | running reward tracker with trend detection |
| `ContinuousLearner` | orchestrates the full loop |

## 1. Setup

In [ ]:
import tempfile
from multigen.learning import (
    FeedbackEntry, FeedbackStore, ExperienceReplay,
    FewShotSelector, AdaptivePrompt, OnlineLearner,
    RewardSignal, ContinuousLearner,
)
tmpdir = tempfile.mkdtemp()
store  = FeedbackStore(f'{tmpdir}/fb.db')   # durable
print('FeedbackStore ready')

## 2. Collecting feedback

In [ ]:
# Simulate 20 agent interactions with varying quality
import random; rng = random.Random(42)

qa_pairs = [
    ('What is transformer architecture?', 'Transformers use self-attention to process sequences in parallel.', 0.92),
    ('Explain gradient descent', 'Gradient descent minimises loss by moving in the direction of the negative gradient.', 0.88),
    ('What is overfitting?', 'When a model memorises training data and fails to generalise.', 0.85),
    ('What is backpropagation?', 'A method to compute gradients via the chain rule.', 0.81),
    ('Explain RLHF', 'Reinforcement Learning from Human Feedback aligns models with human preferences.', 0.79),
    ('What is a vector database?', 'A database that stores high-dimensional embeddings.', 0.74),
    ('Explain cosine similarity', 'It measures the angle between two vectors.', 0.70),
    ('What is LoRA?', 'Low-rank adaptation is a method to fine-tune large models efficiently.', 0.68),
    ('What is RAG?', 'Retrieval Augmented Generation retrieves documents before generation.', 0.65),
    ('Explain embeddings', 'Dense vector representations of text.', 0.60),
]

for q, a, score in qa_pairs:
    label = 'good' if score >= 0.75 else 'neutral'
    correction = None
    if score < 0.65:
        correction = a + ' (more detail needed)'
    store.add(FeedbackEntry(
        input={'question': q},
        output=a,
        score=score,
        label=label,
        correction=correction,
        tags=['qa', 'ml-topic'],
    ))

print(f'Added {len(store)} feedback entries')
print(f'Mean score: {store.mean_score():.3f}')
print(f'Score trend: {store.score_trend():.3f}')

## 3. Experience replay

In [ ]:
replay = ExperienceReplay(capacity=1000)
replay.push_all(store.all())

# Uniform sample
uniform = replay.sample(4)
print('Uniform sample scores:', [round(e.score,2) for e in uniform])

# Prioritised — extreme scores get higher weight
prio = replay.sample_prioritised(4, alpha=0.8)
print('Prioritised sample scores:', [round(e.score,2) for e in prio])

## 4. Few-shot selector

In [ ]:
selector = FewShotSelector(store, n_shots=3, min_score=0.70)

# Select examples relevant to a new query
new_query = {'question': 'How does attention mechanism work in transformers?'}
shots = selector.select(new_query)

print(f'Selected {len(shots)} few-shot examples:')
for i, ex in enumerate(shots, 1):
    print(f'  {i}. [{ex.score:.2f}] Q: {ex.input["question"]}')

print('\nFormatted few-shot block:')
print(selector.format_shots(shots))

## 5. Adaptive prompt

In [ ]:
prompt = AdaptivePrompt(
    base_template='Answer the following machine learning question concisely: {question}',
    store=store,
    n_shots=2,
    include_lessons=True,
)

enriched = prompt.render({'question': 'What is the difference between RNN and LSTM?'})
print('Enriched prompt:')
print('─' * 60)
print(enriched)
print('─' * 60)

## 6. Online learner — epsilon-greedy prompt selection

In [ ]:
learner = OnlineLearner(
    variants={
        'concise':     'Answer in one sentence: {question}',
        'step_by_step': 'Think step by step. {question}',
        'expert':      'You are an ML expert. {question}',
    },
    epsilon=0.1,
    seed=42,
)

# Simulate feedback loop — 'expert' consistently gets better scores
for round_i in range(30):
    name, prompt_str = learner.select_prompt({'question': 'Explain attention'})
    # Mock scores: expert > step_by_step > concise
    score = {'expert': 0.90, 'step_by_step': 0.75, 'concise': 0.60}.get(name, 0.5)
    score += (rng.random() - 0.5) * 0.1  # noise
    learner.record(name, score)

print('Variant performance:')
for variant, stats in learner.stats().items():
    print(f'  {variant:<20}: mean={stats["mean"]:.3f}')
print(f'Best variant: {learner.best_variant()}')

## 7. Reward signal

In [ ]:
reward = RewardSignal('qa_quality', window=50)

# Simulate improving scores over time
for i, (_, _, score) in enumerate(qa_pairs):
    reward.record(score)
    # Simulate improvement after feedback loop
    if i >= 5:
        reward.record(min(1.0, score + 0.1))

print(f'Reward signal stats: {reward.stats()}')
print(f'Mean: {reward.mean():.3f}')
print(f'Trend: {"improving" if reward.trend() > 0 else "degrading"}  ({reward.trend():+.3f})')

## 8. ContinuousLearner — end-to-end

In [ ]:
# Mock agent that uses the adaptive prompt
async def adaptive_agent(ctx: dict) -> dict:
    prompt_obj = AdaptivePrompt(
        'Answer: {question}', store=store, n_shots=2
    )
    rendered = prompt_obj.render(ctx)
    # In production: call real LLM with rendered
    answer = f'Adaptive answer for: {ctx.get("question", "")[:30]}'
    return {'answer': answer, 'prompt_len': len(rendered)}

cl = ContinuousLearner(
    agent_fn=adaptive_agent,
    store=store,
    output_key='answer',
)

# Run several interactions and record feedback
for q, expected_a, expected_score in qa_pairs[:5]:
    result = await cl.run({'question': q})
    entry = await cl.record_feedback(result, score=expected_score, tags=['eval'])

print('ContinuousLearner stats:')
for k, v in cl.stats().items():
    print(f'  {k}: {v}')

## Summary

The continuous learning loop:

```
 1. Select prompt variant   ← OnlineLearner (epsilon-greedy)
 2. Enrich with few-shots   ← FewShotSelector
 3. Run agent               ← ContinuousLearner.run()
 4. Collect feedback        ← ContinuousLearner.record_feedback()
 5. Update reward signal    ← RewardSignal.record()
 6. Sample for replay       ← ExperienceReplay.sample_prioritised()
 7. Loop → agent improves
```

All components work with any store backend:
- `FeedbackStore()` — in-memory (default)
- `FeedbackStore("feedback.db")` — durable SQLite